# Laboratorio 1 · Bitácora

**Nombre:** **Mónica Alejandra Velasco Segura**
**Usuario de GitHub:**  *monicavelasco01-cloud*
**Fecha:** *28/08/2026*

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla que hace que esto sirva de algo:** la predicción se escribe
> *antes* de ejecutar. Si la rellenas después ya sabiendo el resultado, el
> ejercicio no mide nada y tú no aprendes nada. Nadie va a comprobarlo:
> es un trato contigo mismo.


In [15]:
from rlrs.dp import greedy_policy, q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld
from rlrs.evaluation import compare, evaluate
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

# Este cuaderno es una bitácora: no define algoritmos, los usa.
# Si necesitas escribir una función que valga la pena conservar,
# va en src/rlrs/, no aquí.


## Mi variante  (RUIDO = 0.2, COSTE = -0.1)

Ejecuta `uv run python scripts/variante.py` y anota lo que te tocó.


In [16]:
RUIDO  = 0.2   # <- rellena
COSTE  = -0.1   # <- rellena
GAMMA  = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)
mi_env.noise, mi_env.step_reward


(0.2, -0.1)

---
## Ejercicio 1 · El respaldo a mano


**Antes de ejecutar.** ¿Cuál de las cuatro acciones crees que gana en (0,3), y por qué?

_Tu predicción:_ Creo que la que queda de segunda es arriba, porque, aunque chocaría contra el borde superior (lo cual cuesta un paso), mantiene al agente en una posición desde la cual puede intentar llegar a la meta en el siguiente turno. Pienso que no es otra porque creería que tanto "abajo" o "izquierda" desde el dibujo deberian ser mejores porque se alejan de la trampa, pero generan más costos.


In [17]:
from rlrs.dp import q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld

env = GridWorld()                      # el de la guía, no tu variante
valores, politica, barridos = value_iteration(env, gamma=0.9)

for a in range(env.n_actions):
    print(f'{ACTION_NAMES[a]:<10} {q_value(env, valores, (0, 3), a, 0.9):+.6f}')


arriba     +0.800017
derecha    +0.928402
abajo      +0.554337
izquierda  +0.636940


**Resultados:** 
arriba     +0.800017
derecha    +0.928402
abajo      +0.554337
izquierda  +0.636940

**Explica.** ¿Coincidió? ¿Por qué gana esa y no las otras?

_Tu respuesta:_ si, ya que el costo de chocar contra un borde (que equivale a un paso perdido) es significativamente menor que el riesgo de acercarse a la trampa (que penaliza con $-1$). 


---
## Ejercicio 2 · Tu variante, medida


**Antes de ejecutar** ¿tu entorno necesitará más o menos barridos que los 35 del entorno de la guía? Apuesta y di por qué.

**Respuesta:** Yo creo que necesita menos barridos que el que se dio en la guía, ya que un costo por paso más alto ($-0.1$ vs $-0.04$) haría que se redujera el tiempo de convergencia.

In [18]:
from rlrs.evaluation import compare
from rlrs.policies import GreedyTabularPolicy, RandomPolicy
mi_env = GridWorld(noise=0.2, step_reward=-0.1)
valores, politica, barridos = value_iteration(mi_env, gamma=0.9)

print(f'{barridos} barridos · V(3,0) = {valores[mi_env.state_index((3, 0))]:+.4f}')
print(mi_env.render_values(valores, politica))

for r in compare(mi_env, [GreedyTabularPolicy(politica, name='optima'), RandomPolicy(mi_env.n_actions, seed=0)], episodes=300, base_seed=0):
    print(' ', r)

35 barridos · V(3,0) = -0.1281
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 
  optima         retorno +0.151 [+0.109, +0.193]  exito 98.0%  pasos   9.1
  aleatoria      retorno -5.881 [-6.232, -5.530]  exito 25.7%  pasos  56.8


**Coincidio:** No, no coincidio, el entorno convergió en 35 barridos, manteniendo el mismo tiempo de cómputo que la guía, pero con una diferencia crítica en el valor de la salida ($V(3,0) = -0.1281$). La penalización por paso de $-0.1$ reduce significativamente el retorno esperado en comparación con el entorno base ($+0.3440$), lo que refleja que, aunque la política sigue siendo altamente exitosa (98% de éxito), el 'costo de vida' en este entorno es mucho mayor.

Tal vez, no hizo menos barridos, porque el factor de descuento ($\gamma=0.9$) sigue siendo el mismo y es el que domina la velocidad de propagación de la información en la cuadrícula.

**Anota.** Barridos, V(3,0), retorno con su intervalo, tasa de éxito y pasos medios.

_Tus números:_ 

35 barridos · V(3,0) = -0.1281
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 
  optima         retorno +0.151 [+0.109, +0.193]  exito 98.0%  pasos   9.1
  aleatoria      retorno -5.881 [-6.232, -5.530]  exito 25.7%  pasos  56.8

Por tanto: 

Barridos: 35
V(3, 0): -0.1281
Retorno medio: +0.151 (Intervalo: [+0.109, +0.193])
Tasa de éxito: 98.0%
Pasos medios: 9.1

---
## Ejercicio 3 · Subir gamma


**Antes de ejecutar.** Al pasar de 0,9 a 0,99: ¿qué le pasa al número de barridos? ¿Y a la política?

_Tu predicción:_ Al aumentar $\gamma$ de 0.9 a 0.99, espero que los barridos aumenten. Porque un $\gamma$ más alto significa que el agente valora más el futuro, lo que puede llevar a una mayor exploración y, por lo tanto, más iteraciones hasta que converja. Creo que podría esperar un aumento de 10 a 20 barridos. En cuanto a la cantidad de casillas donde la flecha cambia espero que sean pocas, pensaría que cambiaría alrededor de 3 a 5 casillas.




In [19]:
for g in (0.5, 0.9, 0.99):
    v, p, n = value_iteration(mi_env, gamma=g)
    print(f'\ngamma = {g}   {n} barridos   V(3,0) = {v[mi_env.state_index((3, 0))]:+.4f}')
    print(mi_env.render_values(v, p))


gamma = 0.5   20 barridos   V(3,0) = -0.1915
-0.11>  -0.00>  +0.24>  +0.83>   +1    
-0.16^    ###   +0.00^  +0.14^   -1    
-0.18^  -0.16>  -0.11^    ###   -0.20v 
-0.19^  -0.18>  -0.16^  -0.18<  -0.19< 

gamma = 0.9   35 barridos   V(3,0) = -0.1281
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 

gamma = 0.99   40 barridos   V(3,0) = +0.1311
+0.50>  +0.65>  +0.78>  +0.93>   +1    
+0.37^    ###   +0.64^  +0.61^   -1    
+0.25^  +0.34>  +0.49^    ###   -0.07v 
+0.13^  +0.22^  +0.33^  +0.20<  +0.06< 


**Resultados:** 
gamma = 0.5   20 barridos   V(3,0) = -0.1915
-0.11>  -0.00>  +0.24>  +0.83>   +1    
-0.16^    ###   +0.00^  +0.14^   -1    
-0.18^  -0.16>  -0.11^    ###   -0.20v 
-0.19^  -0.18>  -0.16^  -0.18<  -0.19< 

gamma = 0.9   35 barridos   V(3,0) = -0.1281
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 

gamma = 0.99   40 barridos   V(3,0) = +0.1311
+0.50>  +0.65>  +0.78>  +0.93>   +1    
+0.37^    ###   +0.64^  +0.61^   -1    
+0.25^  +0.34>  +0.49^    ###   -0.07v 
+0.13^  +0.22^  +0.33^  +0.20<  +0.06< 


**Explica.** ¿Qué se movió mucho y qué se movió poco? ¿Por qué?

_Tu respuesta:_ Para el valor de la casilla de salida, este cambió significativamente al aumentar $\gamma$. Ya que pasó de -0.1915 ($\gamma$ = 0.5) a +0.1311 ($\gamma$ = 0.99). Esto indica que el agente, al valorar más las recompensas futuras, puede encontrar caminos más óptimos hacia la meta. Para el número de barridos, tenemos que aumentó de 20 ($\gamma$ = 0.5) a 40 (γ = 0.99). Esto confirma que el agente se toma más tiempo para explorar y converger a una política óptima cuando $\gamma$ es más alto, lo que coincide con lo que habia pensado al inicio, y esto es porque un mayor valor de $\gamma$ lleva a una mayor exploración. Aunque, esperaba que al aumentar $\gamma$, los barridos aumentarían debido a que el agente valora más las recompensas futuras, lo que se confirmó con un cambio de 20 a 40 barridos.

En conclusión, el valor de $\gamma$ afecta la toma de decisiones de un agente en un entorno de aprendizaje por refuerzo, esto se comprueba en la politica que tambien cambio notoriamente. Por lo que veo a medida que $\gamma$ se acerca a 1, el agente se vuelve más detalloso, priorizando las recompensas futuras.

---
## Ejercicio 4 · Quitar el ruido


**Antes de ejecutar.** Con ruido 0, ¿cambia la política óptima respecto a la tuya? ¿En qué casillas?

_Tu predicción:_ Espero que cambien en 5 casillas. Ya que, al eliminar el ruido, el agente tendrá un comportamiento un poco más determinista y podrá tomar decisiones más óptimas, lo que probablemente cambiará algunas de las acciones en las casillas cercanas a la meta.


In [20]:
for ruido in (0.0, 0.2, 0.6):
    e = GridWorld(noise=ruido)
    v, p, n = value_iteration(e, gamma=0.9)
    print(f'\nruido = {ruido}   {n} barridos')
    print(e.render_values(v, p))



ruido = 0.0   9 barridos
+0.62>  +0.73>  +0.86>  +1.00>   +1    
+0.52^    ###   +0.73^  +0.86^   -1    
+0.43^  +0.52>  +0.62^    ###   +0.27v 
+0.34^  +0.43^  +0.52^  +0.43<  +0.34< 

ruido = 0.2   35 barridos
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17< 

ruido = 0.6   90 barridos
+0.02^  +0.17>  +0.33>  +0.62>   +1    
-0.08^    ###   +0.21^  +0.28<   -1    
-0.13^  -0.11>  +0.01^    ###   -0.28v 
-0.18^  -0.16>  -0.13<  -0.19<  -0.25v 


**Resultados:** 
ruido = 0.0   9 barridos
+0.62>  +0.73>  +0.86>  +1.00>   +1    
+0.52^    ###   +0.73^  +0.86^   -1    
+0.43^  +0.52>  +0.62^    ###   +0.27v 
+0.34^  +0.43^  +0.52^  +0.43<  +0.34< 

ruido = 0.2   35 barridos
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17< 

ruido = 0.6   90 barridos
+0.02^  +0.17>  +0.33>  +0.62>   +1    
-0.08^    ###   +0.21^  +0.28<   -1    
-0.13^  -0.11>  +0.01^    ###   -0.28v 
-0.18^  -0.16>  -0.13<  -0.19<  -0.25v 

**Explica.** ¿Acertaste? Si te sorprendió, di exactamente qué esperabas y qué pasó.

_Tu respuesta:_ Me da 14 casillas, pero no estoy segura en como interpretar los resultados. 


---
## Ejercicio 5 · Encarecer el paso


**Antes de ejecutar.** Con coste por paso −2, ¿qué hará el agente?

_Tu predicción:_ Espero que la tasa de éxito sea menor con un costo de -2.0 en comparación con -0.04. Esto se debe a que un costo tan alto puede hacer que el agente sea más reacio a moverse, ya que cada acción le costará mucho más. Por lo tanto, es probable que el agente prefiera no actuar o tome decisiones menos arriesgadas, lo que podría resultar en una menor tasa de éxito.


In [21]:
for coste in (-0.001, -0.04, -2.0):
    e = GridWorld(step_reward=coste)
    v, p, n = value_iteration(e, gamma=0.9)
    ev = evaluate(GridWorld(step_reward=coste), GreedyTabularPolicy(p), episodes=300, base_seed=0)
    print(f'coste {coste:>7} · {n:>2} barridos · {ev}')
    print(e.render_values(v, p), '\n')


coste  -0.001 · 41 barridos · avida          retorno +0.991 [+0.991, +0.992]  exito 100.0%  pasos   9.6
+0.62>  +0.71>  +0.82>  +0.94>   +1    
+0.54^    ###   +0.71^  +0.65<   -1    
+0.48^  +0.53>  +0.61^    ###   +0.35v 
+0.42^  +0.47^  +0.52^  +0.46<  +0.40<  

coste   -0.04 · 35 barridos · avida          retorno +0.636 [+0.603, +0.670]  exito 98.0%  pasos   9.1
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17<  

coste    -2.0 · 29 barridos · avida          retorno -13.393 [-13.744, -13.043]  exito  6.0%  pasos   7.3
-6.54>  -4.47>  -2.31>  +0.31>   +1    
-8.18^    ###   -3.66>  -1.29>   -1    
-9.20>  -7.71>  -5.86^    ###   -1.46^ 
-10.11>  -8.85>  -7.44>  -5.90>  -3.94^  



**Resultados:** coste  -0.001 · 41 barridos · avida          retorno +0.991 [+0.991, +0.992]  exito 100.0%  pasos   9.6
+0.62>  +0.71>  +0.82>  +0.94>   +1    
+0.54^    ###   +0.71^  +0.65<   -1    
+0.48^  +0.53>  +0.61^    ###   +0.35v 
+0.42^  +0.47^  +0.52^  +0.46<  +0.40<  

coste   -0.04 · 35 barridos · avida          retorno +0.636 [+0.603, +0.670]  exito 98.0%  pasos   9.1
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17<  

coste    -2.0 · 29 barridos · avida          retorno -13.393 [-13.744, -13.043]  exito  6.0%  pasos   7.3
-6.54>  -4.47>  -2.31>  +0.31>   +1    
-8.18^    ###   -3.66>  -1.29>   -1    
-9.20>  -7.71>  -5.86^    ###   -1.46^ 
-10.11>  -8.85>  -7.44>  -5.90>  -3.94^ 

a. Costo -0.001 
Barridos: 41
Retorno: +0.991 [+0.991, +0.992]
Éxito: 100.0%
Pasos: 9.6

b. Costo -0.04
Barridos: 35
Retorno: +0.636 [+0.603, +0.670]
Éxito: 98.0%
Pasos: 9.1

c. Costo -2.0
Barridos: 29
Retorno: -13.393 [-13.744, -13.043]
Éxito: 6.0%
Pasos: 7.3



**Explica.** ¿Qué está optimizando exactamente el agente para comportarse así?

_Tu respuesta:_  Mi predicción fue que la tasa de éxito con un costo de -2.0 sería menor que con -0.04. El resultado de 6.0% confirma que el agente tuvo muchas dificultades para completar la tarea. Esto no fue sorprendente, ya que un costo tan alto por paso hace que el agente sea extremadamente cauteloso. Así, se está optimizando su retorno esperado, pero con un fuerte enfoque en minimizar las pérdidas debido a un alto costo por paso. 


---
## Ejercicio 6 · El error plantado


**Antes de ejecutar.** en el intento 2, con $\gamma$=1 $\gamma$=1 y coste −0,04, ¿crees que converge o no? Escríbelo.

_Respuesta:_ Creo que no convergerá. Esto se debe a que con $\gamma$=1 (sin descuento), el agente valorará todas las recompensas futuras de manera igual que las inmediatas. Sin embargo, dado que el costo es negativo, esto podría llevar a que el agente no tenga un punto fijo finito, ya que cada acción que toma implica un costo que podría hacer que el valor esperado se vuelva infinito o muy negativo.

  UPTC · Sesion 1 · Que sostiene la convergencia

  1) value_iteration(env, gamma=1.0)
     ValueError: gamma debe estar en [0, 1); se recibio 1.0
     La guardia protege una garantia: sin gamma < 1 el operador de
     Bellman deja de ser una contraccion.

  2) gamma = 1.0, recompensa por paso -0.04  (el entorno de siempre)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1      -0.0400       0.7920     0.792000
         5      -0.2000       0.9513     0.337498
        10       0.4938       0.9685     0.158584
        50       0.6675       0.9721     0.000000
       100       0.6675       0.9721     0.000000
       500       0.6675       0.9721     0.000000
      1000       0.6675       0.9721     0.000000
      2000       0.6675       0.9721     0.000000
     Converge. Quedarse dando vueltas cuesta -0.04 por paso, o sea
     -infinito, asi que ninguna politica que el max prefiera lo hace.
     Es un camino mas corto estocastico, y ahi gamma = 1 esta bien
     definido. Perder la garantia no es perder la convergencia.

  3) gamma = 1.0, recompensa por paso +0.01  (ahora le pagamos por moverse)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1       0.0100       0.8020     0.802000
         5       0.0500       0.9703     0.368426
        10       0.8791       1.0174     0.180948
        50       1.4173       1.4173     0.010000
       100       1.9173       1.9173     0.010000
       500       5.9173       5.9173     0.010000
      1000      10.9173      10.9173     0.010000
      2000      20.9173      20.9173     0.010000
     No converge. A partir del barrido 100 los valores crecen +0.01
     por barrido, indefinidamente, y el cambio se estanca en 0.01:
     el criterio de parada nunca se dispara.

  El mismo entorno con gamma = 0.9 converge en 42 barridos,
  con max|V| = 0.9496. La unica diferencia es el descuento.

  El diagnostico tiene dos capas.
    Matematica: con gamma = 1 existe una politica que nunca termina y
    acumula +0.01 sin fin, luego su retorno es +infinito. No hay punto
    fijo finito al que converger.
    De diseno: el error no fue poner gamma = 1, fue pagar por el
    proceso en vez de por el resultado. El descuento solo lo tapaba.

**Explica por qué coincidió con tu predicción, o por qué no:** Coincidió con los resultados observados, donde el agente no encontró un punto fijo finito debido a la acumulación de costos negativos. La iteración de valor mostró que, aunque convergió a un valor, el costo de permanecer en un estado era significativo, lo que llevó a un comportamiento no óptimo. 


**Explica las dos capas del diagnóstico.**

_La capa matemática:_ Cuando $\gamma$=1, el agente no descuenta las recompensas futuras. Esto significa que puede seguir acumulando recompensa indefinidamente. En el caso de un costo de -0.04, el agente tiene un incentivo para evitar moverse, ya que cada paso le cuesta. Esto implica que no hay un punto fijo finito al que converger, ya que el retorno se vuelve +infinito o -infinito dependiendo de cómo se estructuren las recompensas. La falta de un punto fijo finito es la razón principal por la que la iteración de valor no puede converger de manera efectiva.

_La capa de diseño:_ El problema está en cómo se diseñaron las recompensas, ya que pagar por el proceso (moverse) en lugar de por el resultado (alcanzar la meta) llevó a un comportamiento en el que el agente prioriza la acumulación de recompensas positivas sin considerar el costo de cada acción.


---
## Cierre

**Lo que más me sorprendió hoy:**  Que siempre hay muchas cosas por aprender, porque siempre creemos que lo sabemos todo pero en realidad conoce muy poco, me sorprendio las fórmulas que se presentaron y las aplicaciones que también tienen en esta área que es bastante potente, no las conocia solo algo de la de Markov pero todo chevre y muy entendible.

**Lo que todavía no entiendo:** Me toma mucho tiempo entender los resultados, aunque como matemática, analizo y hasta que no entienda su comportamiento no avanzo, entonces si me cuesta avanzar. Me queda la duda en como contar si se aumenta o disminuye la cantidad de casillas, ya que pues hice varios intentos y al contar me equivoque, asi que es un aspecto que debo mejorar.
